# ColabのL4 GPUでvLLMを立てる

ColabのL4 GPU上のvLLMで `openai/gpt-oss-20b` を起動し、手元から OpenAI 互換API（`/v1/chat/completions` など）を叩けるようにする。RAGをOllamaからvLLMへ移すための動作・速度確認用で、RAGアプリ（`rag_chat_app.py`）はまだOllamaのAPI（`/api/chat` など）を使うため、このノートブックのURLを `.env` に入れてもRAGは動かない。

**手順**
1. ランタイム → ランタイムのタイプを変更 → GPU（L4）を選択してから、上から順にセルを実行する
2. ngrokの無料アカウントを作り、ダッシュボードでauthtokenを控えておく（未取得なら ngrok.com で登録できる）
3. セル7で表示されるURLとAPIキーを使い、末尾の「手元から試す」の手順で呼び出す

**構成**: ngrok → 認証プロキシ（8000番、`X-API-Key` を検証）→ vLLM（8001番）。Ollama版（`run_ollama_server.ipynb`）と同じ認証の流儀にしてあり、手元の `.env` の `OLLAMA_HOST` / `OLLAMA_API_KEY` の形のまま使える。

**注意**
- Colabのランタイムはアイドルや時間経過で切断される。常時稼働のAPIではなく、使う時だけ起動するものとして扱うこと。
- ngrokのURLは誰でも到達できるが、`X-API-Key` ヘッダーが一致しないリクエストはプロキシが401で拒否する。それでもURLとAPIキーは他人に共有しないこと。
- 初回はvLLMの導入（数分）とモデルのダウンロード（約13GB）で時間がかかる。

In [ ]:
# 0. 設定。後続のセルはここの値を使う（値を変えたらこのセルから実行し直す）。
VLLM_VENV = "/content/vllm-env"
VLLM_MODEL = "openai/gpt-oss-20b"
# プロキシが8000番を使うため、vLLMは8001番で待ち受ける
VLLM_PORT = 8001
PROXY_PORT = 8000
# L4のVRAM 24GBを前提にした値。起動時に CUDA out of memory で落ちる場合は
# VLLM_MAX_MODEL_LEN を 32768 に下げる。
VLLM_MAX_MODEL_LEN = 65536
VLLM_GPU_MEMORY_UTILIZATION = "0.90"
# 初回はモデル（約13GB）のダウンロードとCUDAグラフの準備で10分以上かかることがある
VLLM_START_TIMEOUT_SECONDS = 1800

In [ ]:
# 1. GPU（L4）を確認する。
# セル0の値はL4のVRAM 24GBを前提に決めてある。T4（16GB）などが割り当てられると
# vLLMの起動途中で分かりにくい失敗をするため、ここで先に止める。
import subprocess

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_info = gpu_check.stdout.strip()
if gpu_check.returncode != 0 or not gpu_info:
    raise RuntimeError(
        "GPUが見えていません。ランタイムのタイプがGPU（L4）か確認してください: "
        + gpu_check.stderr
    )
if "L4" not in gpu_info:
    raise RuntimeError(
        f"L4以外のGPUが割り当てられています（{gpu_info}）。"
        "ランタイム → ランタイムのタイプを変更 でL4を選び直してください。"
    )
print("GPU確認OK:", gpu_info)

In [ ]:
# 2. vLLMを専用の仮想環境へ入れる（数分かかる）。
# Colab同梱の torch と vLLM が要求する torch のバージョンが食い違うため、
# Colab本体の環境には入れず VLLM_VENV に分ける。--allow-existing は再実行時に
# 既存の仮想環境を作り直さずに使い回すため。
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "venv", VLLM_VENV, "--python", "3.12", "--allow-existing"], check=True)
subprocess.run(
    ["uv", "pip", "install", "--python", f"{VLLM_VENV}/bin/python", "vllm", "--torch-backend=auto"],
    check=True,
)
# FlashInferの事前コンパイル済みカーネルを取得し、起動時のコンパイル待ちを避ける
subprocess.run([f"{VLLM_VENV}/bin/vllm", "download-kernels"], check=True)
print("vLLMの導入OK")

In [ ]:
# 3. vLLMをバックグラウンドで起動し、/v1/models が応答するまで待つ。
# 起動途中で落ちた場合（VRAM不足など）は待ち時間を使い切らず、すぐにログの
# 末尾を表示して止める。ログ全体は vllm.log にある。
import subprocess
import time
from pathlib import Path

import requests

VLLM_LOG_PATH = Path("vllm.log")

# 再実行時、前回のvLLMがGPUメモリと8001番を掴んだままだと新しいプロセスが起動に失敗する
if "vllm_proc" in globals() and vllm_proc.poll() is None:
    vllm_proc.terminate()
    vllm_proc.wait(timeout=60)

VLLM_COMMAND = [
    f"{VLLM_VENV}/bin/vllm",
    "serve",
    VLLM_MODEL,
    "--port",
    str(VLLM_PORT),
    "--max-model-len",
    str(VLLM_MAX_MODEL_LEN),
    "--gpu-memory-utilization",
    VLLM_GPU_MEMORY_UTILIZATION,
]
vllm_log = VLLM_LOG_PATH.open("w", encoding="utf-8")
vllm_proc = subprocess.Popen(VLLM_COMMAND, stdout=vllm_log, stderr=subprocess.STDOUT)


def show_vllm_log_tail():
    vllm_log.flush()
    print("--- vllm.log（末尾） ---")
    print(VLLM_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-3000:])


for _ in range(VLLM_START_TIMEOUT_SECONDS // 5):
    if vllm_proc.poll() is not None:
        show_vllm_log_tail()
        raise RuntimeError(
            f"vLLMが起動途中で終了しました（終了コード {vllm_proc.returncode}）。上のログを確認してください"
        )
    try:
        if requests.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=5).status_code == 200:
            print("vLLM起動確認OK")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    show_vllm_log_tail()
    raise RuntimeError(f"vLLMが{VLLM_START_TIMEOUT_SECONDS}秒以内に応答しませんでした")